# glmax vs statsmodels — correctness check and timing benchmark

Change `RUNTIME` in the next cell to label output rows when switching between CPU, GPU, and TPU runtimes in Colab (`Runtime → Change runtime type`).

In [ ]:
RUNTIME = "CPU"  # change to "GPU" or "TPU" before running on those runtimes

## Setup

In [ ]:
%pip install -q statsmodels
%pip install -q "git+https://github.com/mancusolab/glmax"

In [ ]:
import time

import numpy as np
import statsmodels.api as sm

import jax
import jax.numpy as jnp

import glmax


print("JAX version :", jax.__version__)
print("Devices     :", jax.devices())

## Real-data correctness check

Load the `cpunish` dataset (n=17) from statsmodels, fit a Poisson GLM with both libraries, and compare coefficients. This confirms glmax and statsmodels agree on the same data before the benchmark.

In [ ]:
data = sm.datasets.cpunish.load_pandas()
y_np = data.endog.values.astype(np.float64)
X_np = sm.add_constant(data.exog.values.astype(np.float64))

sm_result = sm.GLM(y_np, X_np, family=sm.families.Poisson()).fit()
print(sm_result.summary())

In [ ]:
X_jax = jnp.array(X_np)
y_jax = jnp.array(y_np)

fitted = glmax.fit(glmax.Poisson(), X_jax, y_jax)

print("Coefficient comparison (statsmodels vs glmax):")
print(f"{'':>4}  {'statsmodels':>14}  {'glmax':>14}  {'diff':>12}")
for i, (sm_b, gl_b) in enumerate(zip(sm_result.params, fitted.params.beta)):
    diff = float(sm_b) - float(gl_b)
    print(f"  b{i}  {float(sm_b):>14.8f}  {float(gl_b):>14.8f}  {diff:>12.2e}")

## Iteration diagnostic

Before benchmarking, check how many IRLS iterations each library takes per problem size.

statsmodels uses a **relative** convergence criterion (change in deviance / deviance). glmax uses an **absolute** criterion (change in NLL). Because NLL scales with n, a fixed absolute tolerance like `1e-3` gets harder to satisfy as n grows — glmax may run far more iterations than statsmodels at large n, which would explain a slowdown that worsens with problem size.

In [ ]:
def make_poisson_data(n: int, p: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    X = np.column_stack([np.ones(n), rng.standard_normal((n, p - 1))])
    beta_true = rng.standard_normal(p) * 0.3
    mu_true = np.exp(X @ beta_true)
    y = rng.poisson(mu_true).astype(np.float64)
    return X.astype(np.float64), y


SIZES = [
    (500, 10),
    (2_000, 20),
    (10_000, 50),
    (50_000, 100),
]

print(f"{'n':>8}  {'p':>5}  {'sm iters':>10}  {'glmax iters':>12}  {'NLL at conv':>14}")
print("-" * 58)
for n, p in SIZES:
    X_np, y_np = make_poisson_data(n, p)
    X_jax = jnp.array(X_np)
    y_jax = jnp.array(y_np)

    sm_res = sm.GLM(y_np, X_np, family=sm.families.Poisson()).fit()
    sm_iters = len(sm_res.fit_history["deviance"]) - 1

    gl_res = glmax.fit(glmax.Poisson(), X_jax, y_jax)
    gl_iters = int(gl_res.num_iters)
    nll = float(gl_res.objective)

    print(f"{n:>8,}  {p:>5}  {sm_iters:>10}  {gl_iters:>12}  {nll:>14.2f}")

## Timing benchmark

**How JAX timing works:** `glmax.fit` is JIT-compiled on the first call for each distinct array shape. We run one warm-up call outside the timed loop so the benchmark measures steady-state execution, not compilation.

In [ ]:
def time_statsmodels(X, y, n_runs: int = 10) -> float:
    family = sm.families.Poisson()
    sm.GLM(y, X, family=family).fit()  # warm-up
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sm.GLM(y, X, family=family).fit()
    return (time.perf_counter() - t0) / n_runs


def time_glmax(X_jax, y_jax, n_runs: int = 10) -> float:
    family = glmax.Poisson()
    # warm-up: triggers JIT compilation
    result = glmax.fit(family, X_jax, y_jax)
    jax.block_until_ready((result.params.beta, result.mu))
    t0 = time.perf_counter()
    for _ in range(n_runs):
        result = glmax.fit(family, X_jax, y_jax)
        jax.block_until_ready((result.params.beta, result.mu))
    return (time.perf_counter() - t0) / n_runs

In [ ]:
N_RUNS = 10

print(f"{'n':>8}  {'p':>5}  {'statsmodels (ms)':>18}  {'glmax (ms)':>12}  {'speedup':>9}  runtime")
print("-" * 72)
for n, p in SIZES:
    X_np, y_np = make_poisson_data(n, p)
    X_jax = jnp.array(X_np)
    y_jax = jnp.array(y_np)

    t_sm = time_statsmodels(X_np, y_np, n_runs=N_RUNS)
    t_gl = time_glmax(X_jax, y_jax, n_runs=N_RUNS)
    speedup = t_sm / t_gl

    print(f"{n:>8,}  {p:>5}  {t_sm * 1e3:>18.2f}  {t_gl * 1e3:>12.2f}  {speedup:>8.1f}x  {RUNTIME}")

## JIT compilation cost

The warm-up call above hides compilation time from the benchmark. This cell measures it explicitly so you can judge whether it is worth paying on first use.

In [ ]:
for n, p in SIZES:
    X_np, y_np = make_poisson_data(n, p)
    X_jax = jnp.array(X_np)
    y_jax = jnp.array(y_np)

    t0 = time.perf_counter()
    result = glmax.fit(glmax.Poisson(), X_jax, y_jax)
    jax.block_until_ready((result.params.beta, result.mu))
    t_compile = time.perf_counter() - t0

    t_steady = time_glmax(X_jax, y_jax, n_runs=N_RUNS)

    print(
        f"n={n:>6,}  p={p:>3}  "
        f"first call (compile+run): {t_compile * 1e3:>8.1f} ms  "
        f"steady-state: {t_steady * 1e3:>8.2f} ms  [{RUNTIME}]"
    )